# 音乐源分离 - pymss WebUI / ComfyUI

This notebook installs [pymss](https://github.com/pymss-project/pymss) (Python Music Source Separation) on Google Colab with GPU, and supports both **pymss WebUI** and **[ComfyUI](https://github.com/comfyanonymous/ComfyUI)** + [comfy-mss](https://github.com/pymss-project/comfy-mss) nodes.

**What you get:** A browser-accessible interface where you can upload audio and separate vocals, drums, bass, and other instruments.

**Requirements:** Colab runtime with GPU (Runtime > Change runtime type > T4 GPU).

> ⚠️ **Known Issue:** WebUI file upload/download may not work correctly through Cloudflare tunnel. This is likely a pymss WebUI frontend issue, not caused by this notebook. Use REST API or Python API as alternative.

---
*Author: [uwuAitry](https://github.com/uwuAitry) | Based on [pymss](https://github.com/pymss-project/pymss) by pymss-project*

## 1. Install dependencies

In [ ]:
#@title 1. Install dependencies { display-mode: "form" }
# ============================================================
# pymss Colab - Step 1: Install
# ============================================================

import torch, subprocess, shutil, os

# ---- Check GPU ----
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime > Change runtime type > GPU.")
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"[GPU] {gpu_name} ({gpu_mem:.1f} GB)")

# ---- Install pymss[server] ----
!pip install -q "pymss[server]"
pymss_ver = subprocess.check_output(["pip", "show", "pymss"], text=True)
pymss_ver = [l.split(":", 1)[1].strip() for l in pymss_ver.splitlines() if l.startswith("Version:")][0]
print(f"[pymss] {pymss_ver}")

# ---- Download cloudflared ----
if not shutil.which("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
cf_ver = subprocess.check_output(["cloudflared", "--version"], text=True).strip()
print(f"[cloudflared] {cf_ver}")

# ---- Install ComfyUI + comfy-mss (optional, for comfyui mode) ----
COMFYUI_DIR = "/content/ComfyUI"
if not os.path.isdir(COMFYUI_DIR):
    !git clone https://github.com/comfyanonymous/ComfyUI.git {COMFYUI_DIR}
!pip install -q -r {COMFYUI_DIR}/requirements.txt
# Install frontend package (required for cloudflare tunnel to serve assets)
!pip install -q comfyui-frontend-package
# comfy-mss custom node
comfy_mss_dir = os.path.join(COMFYUI_DIR, "custom_nodes", "comfy-mss")
if not os.path.isdir(comfy_mss_dir):
    !git clone https://github.com/pymss-project/comfy-mss.git {comfy_mss_dir}
print(f"[comfyui] ComfyUI + comfy-mss installed at {COMFYUI_DIR}")

print("\nInstallation complete. Run the next cell to start the server.")

## 2. Mount Google Drive (optional)

In [ ]:
#@title 2. Mount Google Drive { display-mode: "form" }
# ============================================================
# Mount Google Drive for persistent model config
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
print('[drive] Google Drive mounted at /content/drive')
print('[drive] Model config path: /content/drive/MyDrive/pymss/models.txt')

## 3. Configure & download models

In [ ]:
#@title 3. Configure & download models { display-mode: "form" }
# ============================================================
# Model configuration and download
# ============================================================
# Leave MODELS empty to read from Google Drive config file.

MODELS = "" #@param {type:"string"}
MODEL_SOURCE = "huggingface" #@param ["huggingface", "modelscope"]
DRIVE_MODEL_LIST = "/content/drive/MyDrive/pymss/models.txt" #@param {type:"string"}
MODEL_DIR = "/tmp/pymss_models" #@param {type:"string"}

import os, subprocess, sys

# ---- Parse model list ----
models = []
if MODELS.strip():
    models = [m.strip() for m in MODELS.strip().splitlines() if m.strip()]
    print(f'[models] Got {len(models)} model(s) from form input')
else:
    if os.path.isfile(DRIVE_MODEL_LIST):
        with open(DRIVE_MODEL_LIST) as f:
            models = [m.strip() for m in f if m.strip() and not m.startswith('#')]
        print(f'[models] Loaded {len(models)} model(s) from {DRIVE_MODEL_LIST}')
    else:
        print(f'[models] No models specified and config not found at {DRIVE_MODEL_LIST}')
        print('[models] You can create the file with one model ID per line:')
        print(f'  echo "model-id-1" > {DRIVE_MODEL_LIST}')

if not models:
    print('[models] No models to download. Add models in the form or create the config file.')
else:
    # ---- Install download tools ----
    if MODEL_SOURCE == 'huggingface':
        !pip install -q huggingface_hub
    else:
        !pip install -q modelscope

    os.makedirs(MODEL_DIR, exist_ok=True)
    downloaded = []
    failed = []

    for model_id in models:
        target = os.path.join(MODEL_DIR, model_id.replace('/', '_'))
        if os.path.isdir(target) and os.listdir(target):
            print(f'[models] SKIP (exists): {model_id}')
            downloaded.append(model_id)
            continue
        print(f'[models] Downloading: {model_id} ...')
        try:
            if MODEL_SOURCE == 'huggingface':
                from huggingface_hub import snapshot_download
                snapshot_download(repo_id=model_id, local_dir=target)
            else:
                from modelscope.hub.snapshot_download import snapshot_download as ms_download
                ms_download(model_id, cache_dir=target)
            downloaded.append(model_id)
            print(f'[models] OK: {model_id}')
        except Exception as e:
            failed.append((model_id, str(e)))
            print(f'[models] FAILED: {model_id} - {e}')

    print(f'\n[models] Summary: {len(downloaded)} downloaded, {len(failed)} failed')
    if failed:
        print('Failed models:')
        for mid, err in failed:
            print(f'  - {mid}: {err}')

## 4. Start server

In [ ]:
#@title 4. Start server + tunnel { display-mode: "form" }
# ============================================================
# pymss Colab - Step 2: Start / Stop server + tunnel
# ============================================================
# Click ▶ to start, click ■ to stop everything.

import subprocess, time, urllib.request, sys, threading, os, signal, re, tempfile

# ---- Select mode ----
MODE = "webui" #@param ["webui", "comfyui"]

PORT = 8188 #@param {type:"integer"}
SERVER_LOG = "/tmp/pymss_server.log"
TUNNEL_LOG = "/tmp/cloudflared.log"

def stream_log(proc, log_path):
    """Read log file and print new lines to cell output."""
    last_pos = 0
    while proc.poll() is None:
        try:
            with open(log_path, "r") as f:
                f.seek(last_pos)
                new = f.read()
                if new:
                    print(new, end="", flush=True)
                    last_pos = f.tell()
        except FileNotFoundError:
            pass
        time.sleep(0.5)

def cleanup_all():
    """Terminate all processes and clean up."""
    global server_proc, tunnel_proc, server_log_f, tunnel_log_f
    # Stop tunnel first
    if tunnel_proc and tunnel_proc.poll() is None:
        print("\n[tunnel] Stopping...")
        tunnel_proc.send_signal(signal.SIGTERM)
        try:
            tunnel_proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            tunnel_proc.kill()
            tunnel_proc.wait()
        print("[tunnel] Stopped.")
    # Stop server
    if server_proc and server_proc.poll() is None:
        print("[server] Stopping...")
        server_proc.terminate()
        try:
            server_proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            server_proc.kill()
            server_proc.wait()
        print("[server] Stopped.")
    # Close log file handles
    for f in [server_log_f, tunnel_log_f]:
        if f and not f.closed:
            f.close()
    # Clean up temp log files
    for p in [TUNNEL_LOG]:
        if os.path.exists(p):
            try:
                os.unlink(p)
            except OSError:
                pass

server_proc = None
tunnel_proc = None
server_log_f = None
tunnel_log_f = None

try:
    # ---- Start server ----
    if MODE == "comfyui":
        print("[comfyui] Starting ComfyUI server...")
        server_log_f = open(SERVER_LOG, "w")
        server_proc = subprocess.Popen(
            [sys.executable, "main.py",
             "--listen", "0.0.0.0", "--port", str(PORT),
             "--enable-cors-header"],
            cwd=COMFYUI_DIR,
            stdout=server_log_f, stderr=subprocess.STDOUT,
        )
        server_log_thread = threading.Thread(target=stream_log, args=(server_proc, SERVER_LOG), daemon=True)
        server_log_thread.start()
        for _ in range(180):
            try:
                urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=2)
                print(f"[comfyui] Server ready on port {PORT}.")
                break
            except Exception:
                time.sleep(1)
        else:
            raise RuntimeError("ComfyUI failed to start within 180s.")
    else:
        print("[pymss] Starting WebUI server (model loads on first request)...")
        server_log_f = open(SERVER_LOG, "w")
        server_proc = subprocess.Popen(
            ["pymss", "serve", "--debug",
             "--host", "0.0.0.0", "--port", str(PORT),
             "--device", "auto", "--source", "modelscope", "--webui"],
            stdout=server_log_f, stderr=subprocess.STDOUT,
        )
        server_log_thread = threading.Thread(target=stream_log, args=(server_proc, SERVER_LOG), daemon=True)
        server_log_thread.start()
        for _ in range(120):
            try:
                urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2)
                print(f"[pymss] Server ready on port {PORT}.")
                break
            except Exception:
                time.sleep(1)
        else:
            raise RuntimeError("pymss server failed to start within 120s.")

    # ---- Start tunnel ----
    print("[tunnel] Starting cloudflared...")
    tunnel_log_f = open(TUNNEL_LOG, "w")
    tunnel_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
        stdout=tunnel_log_f, stderr=subprocess.STDOUT,
    )
    tunnel_log_f.close()  # Close handle so we can re-open for reading
    url_re = re.compile(r"(https://[a-z0-9\-]+\.trycloudflare\.com)")
    public_url = None
    start = time.time()
    while time.time() - start < 30:
        try:
            with open(TUNNEL_LOG, "r") as f:
                content = f.read()
            m = url_re.search(content)
            if m:
                public_url = m.group(1)
                break
        except FileNotFoundError:
            pass
        time.sleep(0.5)
    if not public_url:
        raise RuntimeError("Tunnel failed to start within 30s.")

    # Print URL info
    print(f"\n{'='*60}")
    print(f"  Mode:        {MODE}")
    print(f"  Public URL:  {public_url}")
    if MODE == "comfyui":
        print(f"  WebUI:       {public_url}/")
    else:
        print(f"  WebUI:       {public_url}/ui/")
        print(f"  API docs:    {public_url}/docs")
    print(f"{'='*60}\n")
    print("Click ■ to stop.")

    # Block until user clicks stop button
    server_proc.wait()
    print("[server] Process exited.")

except KeyboardInterrupt:
    cleanup_all()

except Exception as e:
    cleanup_all()
    raise

---

### Quick Reference

**Mode switch:** Change `MODE = "webui"` to `MODE = "comfyui"` in Step 4, then re-run Step 4.

**WebUI (pymss):** Open URL → browse catalog → load model → upload audio → separate.

**ComfyUI (pymss):** Open URL → build/load workflow with MSS Separate nodes → queue prompt.

**Model download:** Add model IDs in Step 3 form (one per line), or create `/content/drive/MyDrive/pymss/models.txt` on Google Drive.

**Python API:**
```python
from pymss import MSSeparator
with MSSeparator.from_model_name("bs_roformer_voc_hyperacev2", download=True, device="auto", output_format="wav", store_dirs="results") as sep:
    sep.process_folder("/path/to/audio")
```

**Cleanup:**
```python
cleanup_all()  # or click ■
```